# Salamander Disease Model Design v5

## Model Structure

### Strata
- **Offspring strata:** S (susceptible), I (infected) — no movement between sites
- **Adult strata:** R_a (cleared), R_c (chronic carrier) — movement between sites in pre-season window

### Maturation fates for infected (I) offspring — three branches:
1. Disease death → DEATH (exogenous sink)
2. Chronic carrier → R_c (compartment, can move)
3. Cleared → R_a (compartment, can move)

### Births
- Proportional to total adult population (R_a + R_c)
- Vertical transmission: R_c adults produce infected offspring at rate p_vert
- Birth pulse occurs at the start of the active season (day 135–142)

### Deaths
- Natural mortality out of every compartment
- Higher winter mortality rate for adults; lower summer mortality for both strata

### Seasonality
- Active season: May 15 (day 135) to September 15 (day 258)
- Transmission (beta) is temperature-driven via a Gaussian peaked at 22°C (optimal for ATV)
- Movement occurs in a pre-season window (day 105–135), gated by precipitation threshold and scaled by distance
- Maturation pulse moves all offspring to adult strata at end of season (day 257–259)

### Movement (in progress)
- Adults disperse between ponds in the pre-season window
- Movement probability decays exponentially with distance between ponds
- Precipitation gates and scales movement: below threshold = no movement, above = proportional scaling
- Custom MovementClause required due to use of CustomScope (no built-in geographic support)

In [ ]:
# ========================
# Imports and SSL Setup
# ========================

import certifi
import os
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle
from epymorph.kit import *  # noqa
from pathlib import Path
from sympy import Max
from datetime import date
from epymorph.adrio import acs5, us_tiger, prism as prism_adrio

In [ ]:
# ============================================
# Load Temperature and Precipitation (PRISM)
# ============================================
# Data is cached locally after the first fetch to avoid re-downloading.
# Delete prism_cache.pkl to force a fresh fetch.

current_dir = Path("firstModel.ipynb").resolve().parent
cache_file = current_dir / "data" / "prism_cache.pkl"

if cache_file.exists():
    with open(cache_file, "rb") as f:
        daily_temps, daily_precip = pickle.load(f)
    print("Loaded PRISM data from local cache.")
else:
    coconino_scope = CountyScope.in_counties(["04005"], year=2020)
    temp_time_frame = TimeFrame.range(date(2020, 1, 1), date(2020, 12, 31))
    mean_temp_adrio = prism_adrio.Temperature("Mean")
    precip_adrio = prism_adrio.Precipitation()

    with sim_messaging(live=False):
        temperature_data = mean_temp_adrio.with_context(
            scope=coconino_scope,
            time_frame=temp_time_frame,
            params={"centroid": us_tiger.GeometricCentroid()},
        ).evaluate()

        precip_data = precip_adrio.with_context(
            scope=coconino_scope,
            time_frame=temp_time_frame,
            params={"centroid": us_tiger.GeometricCentroid()},
        ).evaluate()

    daily_temps = temperature_data[:, 0]
    daily_precip = precip_data[:, 0]

    with open(cache_file, "wb") as f:
        pickle.dump((daily_temps, daily_precip), f)
    print("Fetched and cached PRISM data.")

In [ ]:
# =======================
# Load Custom Pond Data
# =======================

test_data_path = current_dir / "data" / "basicTestData.csv"
df = pd.read_csv(test_data_path)

site_ids = df["site_id"].astype(str).tolist()
scope = CustomScope(site_ids)

total_pop = np.array(df["n_salamanders"].tolist(), dtype=int)

seed_location_index = 2
seed_size = int(df.loc[seed_location_index, "initial_infected"])

# One year timeframe
time = TimeFrame.of("2020-01-01", duration_days=365)

adult_frac = 0.40
adult_pop = np.floor(total_pop * adult_frac).astype(int)
offspring_pop = (total_pop - adult_pop).astype(int)

In [ ]:
# ========================
# Compute Distance Matrix
# ========================
# TODO: implement haversine distance matrix from pond lat/lon coordinates
# Requires lat and lon columns in basicTestData.csv
# Will be used by the seasonal movement model below

In [ ]:
# =======================
# Variables for Seasonality
# =======================

season_start = 135  # May 15
season_end = 258    # September 15

In [ ]:
# ================================
# Offspring IPM: S, I + deaths
# ================================

class OffspringSI(CompartmentModel):
    compartments = [
        compartment("S", "susceptible offspring"),
        compartment("I", "infected offspring"),
    ]

    requirements = [
        AttributeDef("beta", type=float, shape=Shapes.TxN,
                     comment="offspring transmission rate"),
        AttributeDef("death_rate", type=float, shape=Shapes.TxN,
                     comment="offspring natural mortality rate"),
        AttributeDef("disease_death_rate", type=float, shape=Shapes.TxN,
                     comment="disease-induced death rate during season")
    ]

    def edges(self, symbols: ModelSymbols) -> list[TransitionDef]:
        S, I = symbols.all_compartments
        beta, mu, disease_death_rate = symbols.all_requirements

        N = Max(1, S + I)

        return [
            # Density-dependent transmission within offspring pool
            edge(S, I, rate=beta * S * I / N),

            # Natural deaths from each compartment
            edge(S, DEATH, rate=mu * S),
            edge(I, DEATH, rate=mu * I),
            edge(I, DEATH, rate=disease_death_rate * I)
        ]

In [ ]:
# ================================================
# Adult IPM: R_a (cleared), R_c (chronic carrier)
# ================================================

class AdultRaRc(CompartmentModel):
    compartments = [
        compartment("R_a", "cleared adult salamanders"),
        compartment("R_c", "chronic carrier adult salamanders"),
    ]

    requirements = [
        AttributeDef("death_rate", type=float, shape=Shapes.TxN,
                     comment="adult natural mortality rate"),
    ]

    def edges(self, symbols: ModelSymbols) -> list[TransitionDef]:
        R_a, R_c = symbols.all_compartments
        mu, = symbols.all_requirements

        return [
            edge(R_a, DEATH, rate=mu * R_a),
            edge(R_c, DEATH, rate=mu * R_c),
        ]

In [ ]:
# ============================
# Multi-Strata Model Builder
# ============================

class SIR_v4(MultiStrataRUMEBuilder):
    def __init__(self):
        self.strata = [
            GPM(
                name="offspring",
                ipm=OffspringSI(),
                mm=mm.No(),  # offspring cannot move between sites
                init=init.NoInfection(),
            ),
            GPM(
                name="adult",
                ipm=AdultRaRc(),
                mm=mm.No(),  # placeholder - custom movement model in progress
                init=init.SingleLocation(
                    initial_compartment="R_a",
                    infection_compartment="R_c",
                    location=seed_location_index,
                    seed_size=seed_size,
                ),
            ),
        ]

        self.meta_requirements = [
            AttributeDef("mature_rate", type=float, shape=Shapes.TxN,
                         comment="end of season maturation"),
            AttributeDef("birth_rate", type=float, shape=Shapes.TxN,
                         comment="births per adult per day"),
            AttributeDef("p_vert", type=float, shape=Shapes.TxN,
                         comment="probability offspring are born infected (vertical transmission)"),
            AttributeDef("p_chronic", type=float, shape=Shapes.TxN,
                         comment="probability infected offspring become chronic carriers on maturation"),
            AttributeDef("p_disease_death", type=float, shape=Shapes.TxN,
                         comment="probability infected offspring die from disease on maturation"),
        ]

    def meta_edges(self, symbols: MultiStrataModelSymbols) -> list[TransitionDef]:
        S, I = symbols.strata_compartments("offspring")
        R_a, R_c = symbols.strata_compartments("adult")

        mature_rate, birth_rate, p_vert, p_chronic, p_disease_death = symbols.all_meta_requirements

        return [
            # End of season: susceptible offspring mature to cleared adults
            edge(S, R_a, rate=mature_rate * S),

            # End of season: infected offspring mature into three fates
            fork(
                edge(I, DEATH, rate=p_disease_death * mature_rate * I),
                edge(I, R_c, rate=p_chronic * mature_rate * I),
                edge(I, R_a, rate=(1 - p_chronic) * mature_rate * I),
            ),

            # Susceptible births: R_a adults + (1-p_vert) fraction of R_c adults
            edge(BIRTH, S, rate=birth_rate * (R_a + ((1 - p_vert) * R_c))),

            # Infected births: p_vert fraction of R_c adults only
            edge(BIRTH, I, rate=birth_rate * (p_vert * R_c)),
        ]

In [ ]:
# ================================================
# Seasonal Beta (Temperature-Driven Transmission)
# ================================================
# Beta follows a Gaussian peaked at 22°C, the optimal temperature for ATV transmission.
# Outside the active season, beta = 0.
# sigma controls how steeply beta drops away from the optimal temperature.

class SeasonalBeta(ParamFunctionTimeAndNode):
    def __init__(self, temps: np.ndarray):
        self.temps = temps

    def evaluate1(self, day: int, node_index: int) -> float:
        beta_max = 0.12
        beta_off = 0.0

        t_mod = day % 365

        if not (season_start <= t_mod <= season_end):
            return beta_off

        temp = self.temps[t_mod]

        optimal_temp = 22.0
        sigma = 5.0

        beta_scaled = beta_max * np.exp(-((temp - optimal_temp) ** 2) / (2 * sigma ** 2))

        return beta_scaled

In [ ]:
# ===============
# Seasonal Births
# ===============
# Birth pulse occurs during the first 7 days of the active season.

class SeasonalBirths(ParamFunctionTimeAndNode):
    def evaluate1(self, day: int, node_index: int) -> float:
        birth_season = 1 / 30
        birth_off = 0.0

        t_mod = day % 365

        if season_start <= t_mod <= season_start + 7:
            return birth_season
        else:
            return birth_off

In [ ]:
# ===============
# Seasonal Deaths
# ===============
# Higher mortality in winter (adults only); lower mortality in summer (adults and juveniles).

class SeasonalDeaths(ParamFunctionTimeAndNode):
    def evaluate1(self, day: int, node_index: int) -> float:
        winter = 1 / 365
        summer = 1 / (365 * 3)

        t_mod = day % 365

        if t_mod >= season_end or t_mod < season_start:
            return winter
        else:
            return summer

In [ ]:
# ==================
# Seasonal Migration (IN PROGRESS)
# ==================
# Custom movement model required — built-in movement models (e.g. mm.Flat) do not support
# CustomScope because they rely on geographic coordinates to compute inter-node distances.
#
# Plan (per Tyler Coles, epymorph lead developer):
#   - Subclass MovementClause directly
#   - Return an NxN matrix per tick describing individuals moving from node i to node j
#   - Movement is gated to a pre-season window (day 105-135)
#   - Distance decay: exponential (e^(-d/scale))
#   - Precipitation: threshold gates movement on/off, then scales proportionally
#
# Known limitation: MovementClause cannot observe current population;
# starting population is used as a proxy (reasonable if movement is small relative to N).
#
# TODO: implement SeasonalAdultMovement once distance matrix is computed

In [ ]:
# ===================
# Seasonal Maturation
# ===================
# All offspring mature to adults over a 3-day window at the end of the season (day 257-259).

class SeasonalMaturation(ParamFunctionTimeAndNode):
    def evaluate1(self, day: int, node_index: int) -> float:
        move_all = 1.0
        season = 0.0

        t_mod = day % 365

        if t_mod >= season_end - 1 and t_mod <= season_end + 1:
            return move_all
        else:
            return season

In [ ]:
# ================
# Build the RUME
# ================

rume = SIR_v4().build(
    scope=scope,
    time_frame=time,
    params={
        # Offspring IPM params
        "gpm:offspring::ipm::beta": SeasonalBeta(daily_temps),
        "gpm:offspring::ipm::death_rate": SeasonalDeaths(),
        "gpm:offspring::ipm::disease_death_rate": 1 / 365,

        # Adult IPM params
        "gpm:adult::ipm::death_rate": SeasonalDeaths(),

        # Meta params
        "meta::ipm::mature_rate": SeasonalMaturation(),
        "meta::ipm::birth_rate": SeasonalBirths(),
        "meta::ipm::p_vert": 0.40,
        "meta::ipm::p_chronic": 0.5,
        "meta::ipm::p_disease_death": 0.20,

        # Populations per strata
        "gpm:offspring::init::population": 0,
        "gpm:adult::mm::population": adult_pop.tolist(),
        "gpm:adult::init::population": adult_pop.tolist(),

        # Adult movement (placeholder until custom movement model is complete)
        "gpm:adult::mm::commuter_proportion": 0.20,
    },
)

In [ ]:
# ==============================
# View Beta and Model Diagram
# ==============================

beta_values = (
    SeasonalBeta(daily_temps)
    .with_context(
        scope=rume.scope,
        time_frame=rume.time_frame,
    )
    .evaluate()
)

fig, ax = plt.subplots()
ax.plot(beta_values)
ax.set(title="Beta Function (Temperature-Driven)", ylabel="beta", xlabel="days")
fig.tight_layout()
plt.show()

# Uncomment to view model compartment diagram
# fig = rume.ipm.diagram()

In [ ]:
# ==================
# Run the Simulation
# ==================

sim = BasicSimulator(rume)
with sim_messaging(live=False):
    out = sim.run(rng_factory=default_rng(5))

df_out = out.dataframe
ponds = out.rume.scope.node_ids

In [ ]:
# ========================
# Plot Simulation Results
# ========================

fig, axes = plt.subplots(len(ponds), 1, figsize=(12, 4 * len(ponds)), sharex=True)

for ax, pond in zip(axes, ponds):
    pond_df = df_out[df_out["node"] == pond]
    ticks = pond_df["tick"].to_numpy()

    ax.plot(ticks, pond_df["S_offspring"].to_numpy(), label="S (offspring)", color="steelblue")
    ax.plot(ticks, pond_df["I_offspring"].to_numpy(), label="I (offspring)", color="firebrick")
    ax.plot(ticks, pond_df["R_a_adult"].to_numpy(), label="R_a (adult)", linestyle="--", color="seagreen")
    ax.plot(ticks, pond_df["R_c_adult"].to_numpy(), label="R_c (adult)", linestyle=":", color="darkorange")

    ax.set_title(f"Pond {pond}")
    ax.set_ylabel("Population")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Day")
plt.suptitle("Salamander Population Dynamics by Pond", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===========
# Diagnostics
# ===========
# Check that maturation works correctly:
# all offspring should transition to adults around day 258.

print("--- Total offspring around maturation (days 254-258) ---")
tot_offspring = df_out.groupby("tick")[["S_offspring", "I_offspring"]].sum()
print(tot_offspring.loc[254:258])

print("\n--- Total adults around maturation (days 254-259) ---")
tot_adults = df_out.groupby("tick")[["R_a_adult", "R_c_adult"]].sum()
print(tot_adults.loc[254:259])

print("\n--- Offspring summary days 250-260 ---")
print(df_out.groupby("tick")[["S_offspring", "I_offspring"]].sum().loc[250:260])